In [2]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

interaction = client.interactions.create(
    model="gemini-4-hydrogen",
    input="Give me a simp-le Indian dal recipe (in less than 10 lines) with its calorie count"
)
print(interaction.output_text)  #unstructured output

**Simple Moong Dal (Serves 2 | ~200 kcal per serving, ~400 kcal total)**

1. Rinse 1/2 cup yellow moong dal and boil with 2 cups water, 1/4 tsp turmeric, and salt for 15 mins until soft.
2. Mash the cooked dal lightly with a spoon until creamy.
3. In a small pan, heat 1 tsp ghee (or oil) and add 1/2 tsp cumin seeds.
4. Once they splutter, sauté 2 minced garlic cloves and a pinch of chili powder for 30 seconds.
5. Pour the tempering over the dal, simmer for 2 minutes, and serve hot.


In [3]:
import json

interaction = client.interactions.create(
    model="gemini-4-hydrogen",
    input="Give me a simp-le Indian dal recipe. Return only json with these keys: title, calories, prep_time_minutes."
)
print(interaction.output_text) 

{
  "title": "Simple Indian Yellow Moong Dal",
  "calories": 220,
  "prep_time_minutes": 10
}


In [9]:
json.loads('''
{
  "title": "Simple Indian Yellow Moong Dal",
  "calories": 220,
  "prep_time_minutes": 10
}
''')

{'title': 'Simple Indian Yellow Moong Dal',
 'calories': 220,
 'prep_time_minutes': 10}

In [10]:
json.loads(interaction.output_text)

{'title': 'Simple Indian Yellow Moong Dal',
 'calories': 220,
 'prep_time_minutes': 10}

### Pydantic more effective way to evaluate particular json format 

In [18]:
from pydantic import BaseModel
from google.genai import types

class Recipe(BaseModel):
    title : str
    ingredients: list[str]
    calories : int
    prep_time_minutes: int

interaction = client.models.generate_content(
    model="gemini-4-hydrogen",
    contents="Give me a simp-le Indian dal recipe",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",  #reply in json
        response_schema=Recipe,                #Excatly the format specified in Recipe class
    ),
)

print(interaction.text)

{"title": "Simple Indian Yellow Dal", "ingredients": ["1 cup split yellow lentils (toor dal)", "3 cups water", "1/2 teaspoon turmeric powder", "1 teaspoon salt", "2 tablespoons ghee or oil", "1 teaspoon cumin seeds", "2 cloves garlic, minced", "1 small onion, chopped", "1 medium tomato, chopped", "1/2 teaspoon red chili powder", "Fresh cilantro leaves for garnish"], "calories": 240, "prep_time_minutes": 30}


In [20]:
recipe = interaction.parsed

In [23]:
print(recipe.calories)
print(recipe.title)

240
Simple Indian Yellow Dal
